# Étape 3: Clustering K-means sur les Word Embeddings

**Objectif:** Appliquer K-means pour grouper les mots similaires

**IMPORTANT:** On utilise la **similarité cosinus** au lieu de la distance euclidienne!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.preprocessing import normalize
import pickle

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

## 1. Charger les données

In [ ]:
# Charger les embeddings originaux (50D)
embedding_matrix = np.load('embedding_matrix.npy')

# Charger les embeddings 2D pour visualisation
embeddings_2d = np.load('embeddings_2d.npy')

# Charger les dictionnaires
with open('word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open('idx2word.pkl', 'rb') as f:
    idx2word = pickle.load(f)

print(f"Embeddings originaux: {embedding_matrix.shape}")
print(f"Embeddings 2D (PCA): {embeddings_2d.shape}")
print(f"Vocabulaire: {len(word2idx)} mots")

## 2. Normalisation pour utiliser la similarité cosinus

**Pourquoi normaliser?**
- K-means utilise la distance euclidienne par défaut
- Pour les word embeddings, la **similarité cosinus** est plus appropriée
- Normaliser les vecteurs permet à K-means d'optimiser indirectement la similarité cosinus

In [ ]:
# Normaliser les embeddings (norme L2 = 1)
# Après normalisation, distance euclidienne ≈ similarité cosinus
embeddings_normalized = normalize(embedding_matrix, norm='l2')

print(f"Embeddings normalisés: {embeddings_normalized.shape}")
print(f"\nVérification: Norme du premier vecteur = {np.linalg.norm(embeddings_normalized[0]):.4f}")
print("(Devrait être ≈ 1.0 après normalisation)")

## 3. Méthode du Coude (Elbow Method) pour choisir K

In [ ]:
# Tester différentes valeurs de K
k_range = range(2, min(11, len(word2idx) // 2))  # De 2 à 10 clusters
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings_normalized)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(embeddings_normalized, kmeans.labels_))

# Visualiser
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Méthode du coude
ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Nombre de Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertie', fontsize=12)
ax1.set_title('Méthode du Coude', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Score de silhouette
ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Nombre de Clusters (K)', fontsize=12)
ax2.set_ylabel('Score de Silhouette', fontsize=12)
ax2.set_title('Score de Silhouette par K', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Trouver le meilleur K selon silhouette
best_k_idx = np.argmax(silhouette_scores)
best_k = list(k_range)[best_k_idx]
print(f"\n✅ Meilleur K selon score de Silhouette: {best_k}")
print(f"   Score de Silhouette: {silhouette_scores[best_k_idx]:.4f}")

## 4. Application du K-means avec le K optimal

In [ ]:
# Vous pouvez changer n_clusters selon votre observation du graphique
n_clusters = best_k  # ou choisir manuellement, ex: n_clusters = 4

# Appliquer K-means
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(embeddings_normalized)

print(f"Clustering avec K = {n_clusters}")
print(f"\nScore de Silhouette: {silhouette_score(embeddings_normalized, cluster_labels):.4f}")
print(f"Score de Calinski-Harabasz: {calinski_harabasz_score(embeddings_normalized, cluster_labels):.2f}")

## 5. Analyse des clusters

In [ ]:
# Organiser les mots par cluster
clusters = {i: [] for i in range(n_clusters)}

for idx, label in enumerate(cluster_labels):
    word = idx2word[idx]
    clusters[label].append(word)

# Afficher les clusters
print("\n" + "="*60)
print("ANALYSE DES CLUSTERS")
print("="*60)

for cluster_id in range(n_clusters):
    words = clusters[cluster_id]
    print(f"\n📌 CLUSTER {cluster_id} ({len(words)} mots):")
    print(f"   Mots: {', '.join(words)}")
    print()

## 6. Visualisation des clusters dans l'espace PCA

In [ ]:
def plot_clusters_pca(embeddings_2d, cluster_labels, idx2word, n_clusters):
    """
    Visualiser les clusters dans l'espace PCA 2D
    """
    plt.figure(figsize=(16, 12))
    
    # Palette de couleurs
    colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
    
    # Tracer chaque cluster
    for cluster_id in range(n_clusters):
        # Points du cluster
        mask = cluster_labels == cluster_id
        cluster_points = embeddings_2d[mask]
        
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                   c=[colors[cluster_id]], label=f'Cluster {cluster_id}',
                   alpha=0.7, s=150, edgecolors='black', linewidth=1.5)
    
    # Ajouter les labels des mots
    for idx, word in idx2word.items():
        x, y = embeddings_2d[idx]
        plt.annotate(word, (x, y),
                    fontsize=10,
                    alpha=0.8,
                    fontweight='bold',
                    xytext=(5, 5),
                    textcoords='offset points')
    
    plt.xlabel('PC1', fontsize=14, fontweight='bold')
    plt.ylabel('PC2', fontsize=14, fontweight='bold')
    plt.title(f'Clustering K-means (K={n_clusters}) - Visualisation PCA', 
             fontsize=16, fontweight='bold')
    plt.legend(loc='best', fontsize=11, framealpha=0.9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_clusters_pca(embeddings_2d, cluster_labels, idx2word, n_clusters)

## 7. Interprétation des Clusters

Analysons chaque cluster pour identifier les thèmes communs:

In [ ]:
def interpret_clusters(clusters, n_clusters):
    """
    Proposer une interprétation pour chaque cluster
    """
    print("\n" + "="*70)
    print("INTERPRÉTATION DES CLUSTERS")
    print("="*70)
    
    interpretations = {}
    
    for cluster_id in range(n_clusters):
        words = clusters[cluster_id]
        
        print(f"\n🔍 CLUSTER {cluster_id}:")
        print(f"   Mots: {', '.join(words)}")
        
        # Suggestions d'interprétation basées sur des patterns communs
        # Vous devez adapter cette partie selon votre corpus
        
        geo_words = {'maroc', 'rabat', 'afrique', 'nord', 'pays'}
        tech_words = {'intelligence', 'artificielle', 'machine', 'learning', 
                     'python', 'programmation', 'réseaux', 'neurones', 'deep'}
        common_words = {'est', 'un', 'une', 'le', 'la', 'de', 'du', 'pour', 'en', 'les'}
        
        word_set = set(words)
        
        if word_set & geo_words:
            interpretation = "Géographie / Localisation"
        elif word_set & tech_words:
            interpretation = "Technologie / Intelligence Artificielle"
        elif word_set & common_words:
            interpretation = "Mots de liaison / Articles"
        else:
            interpretation = "Thème mixte / À déterminer"
        
        interpretations[cluster_id] = interpretation
        print(f"   💡 Interprétation: {interpretation}")
    
    return interpretations

interpretations = interpret_clusters(clusters, n_clusters)

## 8. Calculer la similarité moyenne intra-cluster

In [ ]:
def cosine_similarity(vec1, vec2):
    """Calculer la similarité cosinus"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def cluster_cohesion(embeddings, cluster_labels, n_clusters):
    """
    Calculer la cohésion moyenne de chaque cluster
    (similarité cosinus moyenne entre les mots d'un même cluster)
    """
    print("\n" + "="*60)
    print("COHÉSION DES CLUSTERS (Similarité Cosinus Moyenne)")
    print("="*60)
    
    for cluster_id in range(n_clusters):
        # Indices des mots dans ce cluster
        indices = np.where(cluster_labels == cluster_id)[0]
        
        if len(indices) < 2:
            print(f"\nCluster {cluster_id}: Trop peu de mots pour calculer la cohésion")
            continue
        
        # Calculer toutes les paires de similarités
        similarities = []
        for i in range(len(indices)):
            for j in range(i+1, len(indices)):
                vec1 = embeddings[indices[i]]
                vec2 = embeddings[indices[j]]
                sim = cosine_similarity(vec1, vec2)
                similarities.append(sim)
        
        avg_similarity = np.mean(similarities)
        print(f"\nCluster {cluster_id}: Similarité moyenne = {avg_similarity:.4f}")
        print(f"  (Plus proche de 1 = plus cohésif)")

cluster_cohesion(embedding_matrix, cluster_labels, n_clusters)

## 9. Sauvegarder les résultats

In [ ]:
# Sauvegarder les labels de clusters
np.save('cluster_labels.npy', cluster_labels)

# Sauvegarder le modèle K-means
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

# Sauvegarder les clusters (dictionnaire)
with open('clusters.pkl', 'wb') as f:
    pickle.dump(clusters, f)

# Sauvegarder les interprétations
with open('interpretations.pkl', 'wb') as f:
    pickle.dump(interpretations, f)

print("✅ Résultats du clustering sauvegardés!")
print("\nFichiers créés:")
print("  - cluster_labels.npy")
print("  - kmeans_model.pkl")
print("  - clusters.pkl")
print("  - interpretations.pkl")

## ✅ Étape 3 Complète!

### Récapitulatif du Projet:

**Étape 1:** ✅ Modèle Skip-gram entraîné
**Étape 2:** ✅ Visualisation PCA en 2D
**Étape 3:** ✅ Clustering K-means avec similarité cosinus

### Points Importants:

1. **Similarité Cosinus vs Distance Euclidienne:**
   - Normalisation L2 des embeddings avant K-means
   - Mesure adaptée pour comparer des directions, pas des magnitudes

2. **Choix du nombre de clusters:**
   - Méthode du coude (Elbow)
   - Score de Silhouette
   - Observation visuelle dans l'espace PCA

3. **Interprétation:**
   - Analyser les mots dans chaque cluster
   - Identifier les thèmes communs
   - Calculer la cohésion intra-cluster

### 🎯 Assignment Complet!